In [ ]:
from pathlib import Path
import os, matplotlib.pyplot as plt, numpy as np
from PIL import Image
os.environ.setdefault('MPLCONFIGDIR', str((Path('.cache') / 'matplotlib').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('.cache').resolve()))
I = Path('/Users/jjburrell/Downloads/39-label-images'); O = Path('/Users/jjburrell/Econometrics/Econometrics/final_outputs'); O.mkdir(parents=True, exist_ok=True)
P = sorted(I.glob('*.jpg')); lab = lambda p: 'other' if 'other' in p.stem.lower() else ('cat' if 'cat' in p.stem.lower() else 'dog')
feat = lambda p: (lambda a: (.299*a[...,0] + .587*a[...,1] + .114*a[...,2]).reshape(-1))(np.asarray(Image.open(p).convert('RGB').resize((32,32)), dtype=np.float32) / 255)
soft = lambda z: (lambda e: e / e.sum())(np.exp(z - z.max()))
def probs(X, y, C):
    y = np.asarray(y); out = []
    for i in range(len(X)):
        d = [np.linalg.norm(X[i] - (X[((m := (y == c)).copy() if y[i] == c else m, m.__setitem__(i, False) if y[i] == c else None, m)[0]] if m.any() else X[y == c]).mean(0)) for c in C]
        out.append({c: float(v) for c, v in zip(C, soft(-np.asarray(d)))})
    return out

In [ ]:
R = [{'path': p, 'label': lab(p), 'feature': feat(p)} for p in P]
X = np.stack([r['feature'] for r in R]); y = [r['label'] for r in R]
for r, p in zip(R, probs(X, y, ['cat', 'dog', 'other'])): r['pred'] = max(p, key=p.get)
B = [r for r in R if r['label'] != 'other']; X = np.stack([r['feature'] for r in B]); y = [r['label'] for r in B]
yt = np.array([r['label'] == 'cat' for r in B], int); sc = np.array([p['cat'] for p in probs(X, y, ['cat', 'dog'])])
o = np.argsort(-sc); y, s = yt[o], sc[o]; pos, neg = y.sum(), len(y) - y.sum(); tp = fp = 0; fpr = [0.]; tpr = [0.]; rec = [0.]; pre = [1.]
for i, yi in enumerate(y):
    tp += yi == 1; fp += yi == 0
    if i == len(y) - 1 or s[i + 1] != s[i]: fpr += [fp / neg if neg else 0.]; tpr += [tp / pos if pos else 0.]; rec += [tp / pos if pos else 0.]; pre += [tp / (tp + fp) if tp + fp else 1.]
auc = float(np.trapezoid(tpr, fpr)); ap = float(np.sum(np.diff(rec) * np.array(pre[1:])))
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(fpr, tpr, label=f'AUC = {auc:.3f}'); ax[0].plot([0, 1], [0, 1], '--', c='gray'); ax[0].set(title='ROC for cats', xlabel='False Positive Rate', ylabel='True Positive Rate'); ax[0].legend()
ax[1].plot(rec, pre, label=f'AP = {ap:.3f}', c='tab:orange'); ax[1].set(title='PRC for cats', xlabel='Recall', ylabel='Precision'); ax[1].legend(); fig.tight_layout(); fig.savefig(O / 'cats_roc_prc.png', dpi=200, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(5, 8, figsize=(20, 12)); ax = ax.ravel(); [a.axis('off') for a in ax]
for a, r in zip(ax, R): a.imshow(Image.open(r['path']).convert('RGB')); a.set_title(f"label: {r['label']}, pred: {r['pred']}", fontsize=8)
fig.tight_layout(); fig.savefig(O / 'all_39_images_panel.png', dpi=200, bbox_inches='tight')
print({'roc_auc_cat_vs_dog': auc, 'average_precision_cat_vs_dog': ap})